# VaR Risk Models — Research & Development

Exploratory notebook used during development of the 6-method VaR comparison.
Documents the iterative process: data exploration → model fitting → diagnostics → validation.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import t as student_t, norm, kurtosis, skew

from engine.data_loader import fetch_prices, compute_log_returns
from engine.garch import fit_gjr_garch, simulate_fhs, news_impact_curve, fhs_convergence
from engine.var_models import (
    historical_simulation_var, parametric_normal_var, parametric_student_t_var,
    garch_normal_var, gjr_garch_student_t_var, monte_carlo_fhs_var,
)
from engine.backtest import run_full_backtest, var_exceedance_test

pd.options.display.float_format = '{:.6f}'.format

## 1. Data: SPY 2015–2024

~10 years covers: 2018 vol spike, COVID crash (March 2020), 2022 drawdown.
Enough data for the expanding GARCH window (min 504 obs).

In [ ]:
prices = fetch_prices('SPY', '2015-01-01', '2024-12-31')
returns = compute_log_returns(prices)

print(f'Observations: {len(returns)}')
print(f'Period: {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'\nAnnualised mean:  {returns.mean()*252:.2%}')
print(f'Annualised vol:   {returns.std()*np.sqrt(252):.2%}')
print(f'Skewness:         {skew(returns.values):.3f}')
print(f'Excess kurtosis:  {kurtosis(returns.values, fisher=True):.2f}')
print(f'\nMin daily return: {returns.min():.4f} ({returns.idxmin().date()})')
print(f'Max daily return: {returns.max():.4f} ({returns.idxmax().date()})')

### Fat tails check

If kurtosis >> 3, Normal VaR will systematically underestimate tail risk.
Fit Student-t to get the effective degrees of freedom.

In [ ]:
nu_fit, loc_fit, scale_fit = student_t.fit(returns.values)
print(f'Fitted Student-t: nu = {nu_fit:.2f}')
print(f'  (nu < 10 → heavy tails, nu → inf → converges to Normal)')

# Compare tail quantiles
for q in [0.01, 0.025, 0.05]:
    emp = np.percentile(returns, q*100)
    norm_q = norm.ppf(q, loc=returns.mean(), scale=returns.std())
    t_q = student_t.ppf(q, df=nu_fit, loc=loc_fit, scale=scale_fit)
    print(f'  {q:.1%} quantile — empirical: {emp:.4f}, Normal: {norm_q:.4f}, t({nu_fit:.0f}): {t_q:.4f}')

## 2. GJR-GARCH Fit

Key question: is gamma > 0? (leverage effect = bad news → higher vol)

Checking persistence < 1 (stationarity) and inspecting residual diagnostics.

In [ ]:
result = fit_gjr_garch(returns, distribution='studentt')

print('GJR-GARCH(1,1) with Student-t innovations')
print('=' * 45)
print(f'  omega:       {result.omega:.6f}')
print(f'  alpha:       {result.alpha:.4f}')
print(f'  gamma:       {result.gamma:.4f}  {"← leverage confirmed" if result.gamma > 0.01 else ""}')
print(f'  beta:        {result.beta:.4f}')
print(f'  mu:          {result.mu:.6f}')
print(f'  nu (df):     {result.nu:.1f}')
print(f'  persistence: {result.persistence:.4f}  {"(stationary)" if result.persistence < 1 else "(non-stationary!)"}')
print(f'  half-life:   {result.half_life:.1f} days')
print(f'  AIC:         {result.aic:.1f}')
print(f'  BIC:         {result.bic:.1f}')

### News Impact Curve

If the NIC is asymmetric (steeper on the left), the GJR gamma is doing its job.
This is the key value-add over vanilla GARCH.

In [ ]:
shocks, sigma2 = news_impact_curve(result)

fig = go.Figure()
fig.add_trace(go.Scatter(x=shocks*100, y=sigma2*1e4, mode='lines', line=dict(width=2)))
fig.add_vline(x=0, line_dash='dash', line_color='gray')
fig.update_layout(
    title=f'News Impact Curve (gamma={result.gamma:.4f})',
    xaxis_title='Shock (%)', yaxis_title='sigma^2 (x1e4)',
    template='plotly_white', height=350,
)
fig.show()

## 3. FHS Monte-Carlo

Bootstrap z* from standardised residuals, propagate through GJR filter.
Check convergence — want VaR to stabilise by N~5000.

In [ ]:
fhs = simulate_fhs(result, n_simulations=10000, horizon=1, seed=42)

print(f'FHS results (10k sims, 1-day horizon):')
print(f'  VaR 95%: {fhs.var_95:.4f}   ES 95%: {fhs.es_95:.4f}')
print(f'  VaR 99%: {fhs.var_99:.4f}   ES 99%: {fhs.es_99:.4f}')
print(f'\nFor context: Parametric Normal 99% VaR = {returns.mean() + norm.ppf(0.01)*returns.std():.4f}')
print(f'→ FHS is {abs(fhs.var_99) / abs(returns.mean() + norm.ppf(0.01)*returns.std()):.1f}x more conservative')

In [ ]:
# Convergence diagnostic
n_arr, var_arr = fhs_convergence(result, max_sims=50000, confidence=0.99)

fig = go.Figure()
fig.add_trace(go.Scatter(x=n_arr, y=var_arr, mode='lines+markers', marker=dict(size=3)))
fig.add_hline(y=var_arr[-1], line_dash='dash', line_color='gray')
fig.update_layout(
    title='FHS Convergence: VaR vs N_simulations',
    xaxis_title='N simulations', yaxis_title='VaR 99%',
    template='plotly_white', height=300,
)
fig.show()
print(f'Converged VaR 99% at N=50k: {var_arr[-1]:.5f}')

## 4. Rolling VaR Backtest (all 6 methods)

This is the main comparison. We want to see:
- Normal VaR breached too often (Kupiec rejects)
- GARCH methods adapt to vol regimes
- FHS captures tail risk without being overly conservative

In [ ]:
confidence = 0.99

var_dict = {
    'Historical Simulation': historical_simulation_var(returns, window=252, confidence=confidence),
    'Parametric Normal': parametric_normal_var(returns, window=252, confidence=confidence),
    'Parametric Student-t': parametric_student_t_var(returns, window=252, confidence=confidence),
    'GARCH Normal': garch_normal_var(returns, confidence=confidence, step=5),
    'GJR-GARCH Student-t': gjr_garch_student_t_var(returns, confidence=confidence, step=5),
    'Monte-Carlo FHS': monte_carlo_fhs_var(returns, confidence=confidence, n_sims=2000, step=5),
}

print('Done — all 6 methods computed.')

In [ ]:
bt = run_full_backtest(returns, var_dict, confidence)
bt_display = bt[['Method', 'n_exceedances', 'exceedance_rate', 'expected_rate', 
                 'ratio', 'kupiec_p', 'christoffersen_p', 'combined_p']].copy()
bt_display['pass_kupiec'] = bt_display['kupiec_p'].apply(lambda p: 'PASS' if p > 0.05 else 'FAIL')
bt_display['pass_combined'] = bt_display['combined_p'].apply(lambda p: 'PASS' if p > 0.05 else 'FAIL')
bt_display

## 5. Observations & Design Decisions

Key findings from this analysis:

1. **Parametric Normal always underestimates tail risk** — excess kurtosis ≈ 10-15 for SPY, so Gaussian tails are too thin. Kupiec test confirms this (p < 0.05 = too many breaches).

2. **GJR-GARCH captures the leverage effect** — gamma consistently > 0.1 for equity indices. The NIC asymmetry is visible and significant.

3. **FHS convergence** — VaR stabilises around N=5000 sims. Using 2000 for the rolling backtest is a reasonable speed/accuracy tradeoff; 10000 for the full-sample deep dive.

4. **Step parameter for rolling GARCH** — refitting every 5 days keeps runtime under 30s for 10yr of data while barely affecting accuracy vs daily refit.

5. **Student-t df (nu)** — typically 5-8 for equity indices, confirming heavy tails. The arch library's MLE is robust here.

These findings drove the dashboard design: the Theory tab explains *why* Normal fails, the GARCH tab shows *how* the model adapts, and the FHS tab demonstrates the MC approach.

In [ ]:
# Quick visual: returns + GJR-GARCH VaR + breaches
var_gjr = var_dict['GJR-GARCH Student-t'].dropna()
exc = var_exceedance_test(returns, var_gjr, confidence)
breach_dates = exc['exceedances'][exc['exceedances']].index

fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.values, mode='lines',
                         line=dict(color='gray', width=0.4), opacity=0.5, name='Returns'))
fig.add_trace(go.Scatter(x=var_gjr.index, y=var_gjr.values, mode='lines',
                         line=dict(color='#d62728', width=1.2), name='GJR-t VaR 99%'))
fig.add_trace(go.Scatter(x=breach_dates, y=returns.loc[breach_dates].values,
                         mode='markers', marker=dict(color='red', size=6), name=f'Breaches ({len(breach_dates)})'))
fig.update_layout(title=f'GJR-GARCH Student-t VaR — {len(breach_dates)} breaches / {exc["n_obs"]} obs',
                  template='plotly_white', height=400)
fig.show()